# 1. Đọc dữ liệu và kiểm tra khả năng áp dụng IQR

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Tìm dataset gốc
DATA_PATH = None
SEARCH_ROOTS = [Path.cwd().resolve(), *Path.cwd().resolve().parents]

for folder in SEARCH_ROOTS:
    candidates = [
        folder / "data" / "raw" / "RT_IOT2022.csv",
        folder / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
        folder / "final_IoT" / "sample_project" / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH is not None:
        break

if DATA_PATH is None:
    raise FileNotFoundError("Khong tim thay RT_IOT2022.csv trong cau truc project")

SAMPLE_ROOT = next(
    (
        candidate
        for folder in SEARCH_ROOTS
        for candidate in (folder, folder / "final_IoT" / "sample_project")
        if (candidate / "project").is_dir() and (candidate / "result").is_dir()
    ),
    None,
)
if SAMPLE_ROOT is None:
    raise FileNotFoundError("Khong tim thay thu muc goc sample_project")

RUN_DIR = SAMPLE_ROOT / "result" / "eda" / "result_03_outliers_correlation"
FIG_DIR = RUN_DIR
TABLE_DIR = RUN_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

# Lấy các thuộc tính số, bỏ cột chỉ số
num = df.select_dtypes(include=np.number).drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# Tính các phân vị
q1 = num.quantile(0.25)
q3 = num.quantile(0.75)
iqr = q3 - q1

iqr_summary = pd.DataFrame({
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr,
    "Zero (%)": num.eq(0).mean() * 100,
    "Unique Values": num.nunique()
})

iqr_summary["IQR = 0"] = iqr_summary["IQR"] == 0

print("=== KIỂM TRA IQR ===")
print("Tổng đặc trưng số:", num.shape[1])
print("Số cột IQR > 0:", (iqr > 0).sum())
print("Số cột IQR = 0:", (iqr == 0).sum())

print("\n=== CÁC CỘT IQR = 0 ===")
display(
    iqr_summary[
        iqr_summary["IQR = 0"]
    ].round(4)
)

iqr_summary.to_csv(
    TABLE_DIR / "03_iqr_summary.csv",
    encoding="utf-8-sig"
)

# 2. Phát hiện ngoại lệ bằng IQR

In [ ]:
valid_cols = iqr[iqr > 0].index

lower_bound = q1[valid_cols] - 1.5 * iqr[valid_cols]
upper_bound = q3[valid_cols] + 1.5 * iqr[valid_cols]

# Đánh dấu giá trị nằm ngoài khoảng IQR
outlier_mask = (
    num[valid_cols].lt(lower_bound) |
    num[valid_cols].gt(upper_bound)
)

outlier_count = outlier_mask.sum()

outlier_result = pd.DataFrame({
    "Q1": q1[valid_cols],
    "Q3": q3[valid_cols],
    "IQR": iqr[valid_cols],
    "Lower Bound": lower_bound,
    "Upper Bound": upper_bound,
    "Outlier Count": outlier_count,
    "Outlier (%)": outlier_count / len(df) * 100
})

outlier_result = outlier_result.sort_values(
    "Outlier Count",
    ascending=False
)

print("=== NGOẠI LỆ THEO IQR ===")
display(outlier_result.round(4))

print("\n=== TOP 10 CỘT CÓ NHIỀU GIÁ TRỊ NGOẠI LỆ ===")

display(
    outlier_result[
        ["Outlier Count", "Outlier (%)"]
    ].head(10).round(2)
)

# Vẽ số lượng ngoại lệ
top10 = outlier_result.head(10)

plt.figure(figsize=(11, 6))

sns.barplot(
    x=top10["Outlier Count"].values,
    y=top10.index,
    color="steelblue"
)

plt.title("Top 10 đặc trưng có nhiều ngoại lệ theo IQR")
plt.xlabel("Số lượng giá trị nằm ngoài ngưỡng IQR")
plt.ylabel("Đặc trưng")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_iqr_outlier_counts.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

outlier_result.to_csv(
    TABLE_DIR / "03_iqr_outlier_results.csv",
    encoding="utf-8-sig"
)

# 3. Boxplot: trực quan hóa ngoại lệ

In [ ]:
selected_features = [
    "flow_duration",
    "flow_iat.tot",
    "active.max",
    "flow_pkts_payload.std"
]

fig, axes = plt.subplots(
    2, 2,
    figsize=(15, 9)
)

for ax, col in zip(axes.flat, selected_features):

    sns.boxplot(
        x=num[col],
        ax=ax,
        showfliers=True,
        flierprops={
            "marker": ".",
            "markersize": 1,
            "alpha": 0.3
        }
    )

    # Chỉ đổi thang trục để quan sát rõ hơn
    ax.set_xscale(
        "symlog",
        linthresh=max(float(iqr[col]), 1e-6)
    )

    ax.set_title(f"Boxplot - {col}")
    ax.set_xlabel(col)

plt.suptitle(
    "EDA 03 - Phân tích ngoại lệ bằng Boxplot",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_boxplot_outliers.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

# 4. Correlation Analysis: Pearson và Spearman

In [ ]:
# Loại cột hằng khỏi phép tính tương quan
# Không xóa cột khỏi dataset gốc
corr_data = num.loc[
    :, num.nunique(dropna=False) > 1
]

print("Số đặc trưng dùng tính tương quan:",
      corr_data.shape[1])

# Tính Pearson
pearson_corr = corr_data.corr(method="pearson")

# Tính Spearman
spearman_corr = corr_data.corr(method="spearman")

# Chỉ lấy mỗi cặp một lần, bỏ đường chéo
mask = np.triu(
    np.ones(pearson_corr.shape, dtype=bool),
    k=1
)

pearson_pairs = (
    pearson_corr
    .where(mask)
    .stack()
    .rename("Pearson")
)

spearman_pairs = (
    spearman_corr
    .where(mask)
    .stack()
    .rename("Spearman")
)

# Gộp kết quả
corr_pairs = pd.concat(
    [pearson_pairs, spearman_pairs],
    axis=1
).reset_index()

corr_pairs.columns = [
    "Feature 1",
    "Feature 2",
    "Pearson",
    "Spearman"
]

corr_pairs["Abs Pearson"] = (
    corr_pairs["Pearson"].abs()
)

corr_pairs["Abs Spearman"] = (
    corr_pairs["Spearman"].abs()
)

print("=== PHÂN TÍCH TƯƠNG QUAN ===")

print(
    "Số cặp |Pearson| >= 0.95:",
    (corr_pairs["Abs Pearson"] >= 0.95).sum()
)

print(
    "Số cặp |Spearman| >= 0.95:",
    (corr_pairs["Abs Spearman"] >= 0.95).sum()
)

print("\n=== TOP 20 CẶP PEARSON CAO NHẤT ===")

display(
    corr_pairs.sort_values(
        "Abs Pearson",
        ascending=False
    ).head(20).round(6)
)

# Lưu bảng kết quả
corr_pairs.to_csv(
    TABLE_DIR / "03_correlation_pairs.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5. Correlation Heatmap và kiểm tra cặp đặc trưng tương quan cao

In [ ]:
# Chọn 10 cặp có Pearson tuyệt đối lớn nhất
top_pairs = corr_pairs.sort_values(
    "Abs Pearson",
    ascending=False
).head(10)

# Lấy danh sách cột xuất hiện trong 10 cặp
selected_corr_features = list(dict.fromkeys(
    top_pairs["Feature 1"].tolist()
    + top_pairs["Feature 2"].tolist()
))

print("Các đặc trưng được chọn:")
print(selected_corr_features)

fig, axes = plt.subplots(
    1, 2,
    figsize=(19, 8)
)

# Pearson Heatmap
sns.heatmap(
    pearson_corr.loc[
        selected_corr_features,
        selected_corr_features
    ],
    ax=axes[0],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    annot=False
)

axes[0].set_title("Pearson Correlation Heatmap")

# Spearman Heatmap
sns.heatmap(
    spearman_corr.loc[
        selected_corr_features,
        selected_corr_features
    ],
    ax=axes[1],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    annot=False
)

axes[1].set_title("Spearman Correlation Heatmap")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_correlation_heatmap.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

# Kiểm tra chi tiết một cặp đặc trưng
feature_a = "flow_duration"
feature_b = "flow_iat.tot"

print("\n=== KIỂM TRA CẶP ĐẶC TRƯNG ===")

print("Feature 1:", feature_a)
print("Feature 2:", feature_b)

print(
    "Pearson:",
    pearson_corr.loc[feature_a, feature_b]
)

print(
    "Spearman:",
    spearman_corr.loc[feature_a, feature_b]
)

# Quan sát quan hệ bằng Scatterplot
sample = df[
    [feature_a, feature_b]
].sample(
    n=min(10000, len(df)),
    random_state=42
)

plt.figure(figsize=(9, 6))

plt.scatter(
    sample[feature_a],
    sample[feature_b],
    s=8,
    alpha=0.3
)

# Đổi thang hiển thị để dễ quan sát
plt.xscale("symlog", linthresh=1e-6)
plt.yscale("symlog", linthresh=1)

plt.title(
    "Quan hệ giữa flow_duration và flow_iat.tot"
)

plt.xlabel(feature_a)
plt.ylabel(feature_b)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_high_correlation_scatter.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()